# 07 – v2 Run Experiments

Runs the five v2 stages from `RESULTS_AND_PLAN.md` (Part B, section B4) and stops at each decision point. Each stage cell calls `cluster/submit.sh`:

- **`BACKEND = "local"`**: shards run in parallel on this machine's GPUs (`LOCAL_GPUS` × `PROCS_PER_GPU`), and the cell blocks until the stage finishes. Use it on a workstation or the DGX Spark.
- **`BACKEND = "slurm"`**: the cell submits array jobs chained with `afterok` and returns immediately. Use the progress cells to follow them, and continue once they complete.

Everything is **resumable**: re-running a stage skips finished images and retries failed ones. Do not edit configs between the stages of one split; the manifests record the git state of every shard.

| Stage | Cell | Decision after it |
|---|---|---|
| 1 pilot (dev) | §1 | D1: G1 direction, budget, lite method list |
| 2 calib + τ | §2 | write τ into `eval_config.yaml`, commit |
| 3 test | §3 | G1–G3 and gate B (confirmatory) |
| 4 BDD100K | §4 | transfer and subgroups |
| 5 v1 rows | §5 | comparison rows only |

Prerequisite: notebook 06 checklist complete.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src import notebook_utils as nbu

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

ENV_FILE = "cluster/cluster.env"
BACKEND = None        # None = value in cluster.env; or "local" / "slurm"
DRY_RUN = False       # True: print the submit commands only

env = nbu.load_cluster_env(ENV_FILE)
print("backend:", BACKEND or env.get("BACKEND"), "| GPUs:", env.get("LOCAL_GPUS"), "| partition:", env.get("SLURM_PARTITION"))

def runs(split, domain="coco"):
    return [nbu.run_name(split, key, domain) for key in ("yolox", "frcnn")]

def stage(*names):
    return nbu.submit(names, backend=BACKEND, env_file=ENV_FILE, dry_run=DRY_RUN)

## 1. Pilot — COCO dev, full config

YOLOX runs on all 300 dev images; Faster R-CNN on `NUM_IMAGES_dev_frcnn`. The VLM pass and the CPU analysis follow automatically.

In [ ]:
stage("pilot", "vlm-pilot", "analyze-pilot")

In [ ]:
display(nbu.evidence_progress(runs("dev")))

### D1 — decisions from the pilot

1. **G1 direction.** Read the table below. The pilot is *not* confirmatory, so do not tune the gate. If ΔAUROC(S3 − S2) is ≤ 0 on both detectors, stop and write up gate B only (see `RESULTS_AND_PLAN.md` B1).
2. **Budget.** The projections use the pilot's measured cost.
3. **Lite method list.** Keep methods with `keep_for_lite = True`, preferring a high `gain_per_second`. Edit `evidence.methods` / `agreement_pairs` / `save_maps_for` in `config/evidence_config_lite.yaml`, then **commit** before stage 3. Also decide whether to switch D-CLOSE to 2000 masks (`dclose_final_masks`).

In [ ]:
dev_analysis = nbu.ANALYSIS_ROOT / "evidence_coco_dev"
if (dev_analysis / "summary.json").exists():
    display(nbu.gate_summary(dev_analysis))
    for run in runs("dev"):
        if (dev_analysis / f"g2_method_value_{run}.csv").exists():
            display(Markdown(f"**{run}** — value of each method added alone to output + gradient"))
            display(nbu.suggest_lite_methods(dev_analysis, run).round(4))
    display(Markdown((dev_analysis / "report.md").read_text()))
else:
    print("Pilot analysis not found yet:", dev_analysis)

projections = {}
for run, key in zip(runs("dev"), ("yolox", "frcnn")):
    if nbu.timing_summary(run).get("images"):
        test_images = int(env.get(f"NUM_IMAGES_test_{key}") or 2000)
        projections[f"test {key}"] = nbu.estimate_gpu_hours(run, test_images, max_explained=10, cost_scale=0.8)
if projections:
    display(pd.DataFrame(projections).T)

## 2. Calibration split and τ

This stage runs G-CAME only, since the calib split exists only to set the per-detector escalation τ (p30). The next cell prints the change to `config/eval_config.yaml` and writes it only when `APPLY_TAU = True`. Afterwards, update the "PROVISIONAL" comment above the values and commit.

In [ ]:
stage("calib", "calibrate-tau")

In [ ]:
APPLY_TAU = False
tau_json = nbu.ANALYSIS_ROOT / "tau_calibration.json"
if tau_json.exists():
    print(tau_json.read_text())
    print(nbu.apply_tau_to_eval_config(tau_json.relative_to(nbu.PROJECT_ROOT), dry_run=not APPLY_TAU))
else:
    print("Run the calib stage first.")

## 3. Test — confirmatory run

**Freeze first.** Commit the lite config, the gate thresholds (`analysis` section) and `eval_config.yaml`. The check below refuses to submit while they have uncommitted changes. This is what makes G1–G3 confirmatory rather than tuned on the test set.

In [ ]:
FORCE_UNFROZEN = False   # only for debugging; never for reported runs
if nbu.config_is_committed() or FORCE_UNFROZEN:
    stage("test", "vlm-test", "analyze-test")
else:
    print("Commit the frozen inputs, then re-run this cell.")

In [ ]:
display(nbu.evidence_progress(runs("test")))

## 4. BDD100K — transfer

This needs `BDD_ANN` and `BDD_MANIFEST`, which notebook 06 section 3 creates when the labels are present. The runs use the same frozen lite config.

In [ ]:
if (nbu.PROJECT_ROOT / env["BDD_MANIFEST"]).exists():
    stage("bdd", "vlm-bdd", "analyze-bdd")
else:
    print("No BDD100K manifest; skip or see notebook 06 section 3.")

## 5. v1 comparison rows

These rows run on the first `LEGACY_NUM_IMAGES` test images:
- The relabelled selector checkpoint (`selector`).
- The four fixed baselines × 2 detectors, including the never-run Faster R-CNN LIME.
- The v1 pipeline with feedback and VLM off (`v1`), and with the original settings (`v1orig`).
- The G-CAME→LIME cascade.

Run the cascade **only after** τ was applied in §2, since it uses `tau_by_detector` from `eval_config.yaml`.

In [ ]:
stage("selector", "legacy")

In [ ]:
import yaml

escalation = yaml.safe_load((nbu.PROJECT_ROOT / "config/eval_config.yaml").read_text())["evaluation"]["feedback"]["escalation"]
provisional = {"yolox-s": 0.6546, "fasterrcnn_resnet50_fpn_v2": 0.6922}
if escalation.get("tau_by_detector") == provisional:
    print("tau_by_detector still has the provisional values - apply the calibrated tau (section 2) first.")
else:
    stage("cascade")

## Status of everything

In [ ]:
all_runs = runs("dev", "smoke") + runs("dev") + runs("calib") + runs("test") + runs("test", "bdd")
display(nbu.evidence_progress(all_runs))
legacy = sorted((nbu.PROJECT_ROOT / "results" / "v2_legacy").glob("**/*.csv"))
print(f"{len(legacy)} legacy CSVs under results/v2_legacy")